<a href="https://colab.research.google.com/github/zlwym/Mann_Zoe_LabTask/blob/main/labtask02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 📚 Lab Task 2: Cleaning Up the Mess

You’ll be working with a dataset of real student grades — 7 assignments and a final exam — but things aren’t as clean as they should be. Some values are missing, some are way off, and it’s your job to fix it.

You’ll explore the data, figure out what went wrong, and try different strategies to clean it up.

Get ready to:
- Spot broken data
- Try out different fixes
- Compare models
- Justify your decisions

### Dataset Introduction

The dataset comes from real student grades in a course at SFU. Students completed **7 assignments**, and we also have their **final exam grade**.

It’s your job to explore the dataset and clean it up.

---

> 💡 **Note**: Students could receive bonus marks for some assignments:
> - **A2**: up to **15** points
> - **A4**: up to **5** points
> - **A6**: up to **10** points  
> Keep this in mind when you're evaluating high or unusual scores — they might not be errors!


**Attention:** The bonus values are in **points** not **percentages**!!!
---

### ✅ What You Need to Do

-  **Explore the dataset**
  - Look at basic stats, column names, and what the data looks like
  - Identify anything that stands out right away

-  **Check the correlations**
  - Use a correlation matrix to find relationships between assignments and the final exam
  - Do any assignments seem strongly related to final exam performance?

-  **If you could only use two assignment grades to predict the final exam**, which ones would you choose — and why?

-  **Check for missing values**
  - Which columns have them?
  - How many are missing?

-  **Handle the missing values**
  - Try out different imputation strategies (mean, median, remove, etc.)
  - Which one gives you the best results? Why do you think that is?
  - Exploration idea: search and see what are the ways of evaluating your results. How can you make sure that a strategy for handling the missing values works better than the other?

-  **Check for outliers**
  - Identify values that seem unrealistic or suspicious
  - Decide whether to keep, modify, or remove them — and explain your reasoning
  - Compare the results

---

For each step, be ready to explain your decisions. There isn’t always one "right" answer — we’re more interested in your reasoning!

> 💡 **Note**: If handling missing values and outliers for **all 7 assignments** feels overwhelming, it’s totally fine to **focus on just the two columns you think are most important**.  
> Just make sure your reasoning for choosing them is solid and clearly explained.


# **Loading Dataset**

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("grades_crpt.csv")

In [ ]:
df.head()

,A1,A2,A3,A4,A5,A6,A7,Final_Exam,user_id
0,NaN,NaN,30.0,75.0,90.0,65.0,50.6,68.8,U001
1,100.0,NaN,NaN,92.5,100.0,100.0,84.4,50.3,U002
2,75.0,69.6,NaN,86.2,100.0,NaN,NaN,67.8,U003
3,25.0,78.6,40.0,0.0,50.0,30.7,0.0,0.0,U004
4,0.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,U005


# **Dataset Exploration**

The data seems to be full of null (missing, NaN) values, maybe from missing assignments.

There are also some negative values and values above 100 (which may or may not be due to bonus points).

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   A1          57 non-null     float64
 1   A2          61 non-null     float64
 2   A3          62 non-null     float64
 3   A4          77 non-null     float64
 4   A5          61 non-null     float64
 5   A6          67 non-null     float64
 6   A7          76 non-null     float64
 7   Final_Exam  86 non-null     float64
 8   user_id     86 non-null     object 
dtypes: float64(8), object(1)
memory usage: 6.2+ KB


The final exam and user ID columns are the only ones with no null values.

All assignments, columns A1, A2, A3, A4, A5, A6, and A7, have missing values.

A1: 29 missing values

A2: 25 missing values

A3: 24 missing values

A4: 9 missing values

A5: 25 missing values

A6: 19 missing values

A7: 10 missing values

In [ ]:
df.describe()

,A1,A2,A3,A4,A5,A6,A7,Final_Exam
count,57.000000,61.000000,62.000000,77.000000,61.000000,67.000000,76.000000,86.000000
mean,83.671930,81.096721,68.174194,82.332468,89.645902,74.437313,78.130263,55.509302
std,34.286481,28.556721,42.343621,39.101984,25.510505,31.176535,26.412058,18.176777
min,-4.500000,-30.600000,-70.100000,-21.600000,7.700000,-17.000000,0.000000,0.000000
25%,75.800000,64.300000,50.000000,72.000000,87.000000,60.000000,70.300000,45.850000
50%,87.500000,91.100000,80.000000,87.500000,95.000000,80.000000,80.000000,56.050000
75%,95.800000,100.000000,93.250000,95.000000,100.000000,92.500000,87.500000,67.725000
max,174.600000,148.900000,152.200000,188.200000,173.900000,183.600000,150.600000,97.500000


The minimum values for A1, A2, A3, A4, and A6 are negative values which do not make sense for the context of this dataset.

There are also maximums of more than 100 on A1, A3, A5, and A7 when there were only bonus marks on A2, A4, and A6. There should only be values over 100 in columns A2, A4, and A6.

In [ ]:
neg = df[["A1", "A2", "A3", "A4", "A5", "A6", "A7", "Final_Exam"]].lt(0).any(axis=1)
display(df[neg])

,A1,A2,A3,A4,A5,A6,A7,Final_Exam,user_id
9,129.9,76.8,-5.3,93.8,100.0,90.0,12.2,65.3,U010
12,54.2,92.9,-69.3,81.2,NaN,183.6,82.5,71.9,U013
26,NaN,NaN,-1.7,88.8,55.0,20.0,77.5,33.1,U027
44,-4.5,148.9,100.0,98.8,92.0,NaN,91.5,82.5,U045
52,41.7,58.9,-70.1,88.8,80.0,70.0,71.9,31.9,U053
67,90.0,100.0,83.0,-21.6,97.0,100.0,81.2,67.5,U068
76,83.3,64.3,55.0,78.8,100.0,-17.0,78.8,70.9,U077
82,87.5,-30.6,70.0,174.8,91.0,NaN,146.4,64.1,U083


lt() helps check for values in the dataframe that are less than a specified value and returns True or False. any() helps return a value for each column.